In [ ]:

import gc
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

# Paths and settings
EMB_DIR = "/"
OUTPUT_DIR = "/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EPOCHS = 500
EARLY_STOP = True
PATIENCE = 20
MIN_DELTA = 1e-5
TRAIN_RATIO = 0.8
BATCH_SIZE = 1024
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_float32_matmul_precision("high")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True


def binarize_insider_labels(y):
    y = np.asarray(y)
    return np.where(np.isin(y, [1, 2, 3, 4]), 1, y).astype(int)


def get_hidden_dims(input_dim):
    if input_dim >= 800:
        return [512, 256, 128], 64
    elif input_dim >= 400:
        return [256, 128, 64], 32
    else:
        return [128, 64], 16


class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        hidden_dims, latent_dim = get_hidden_dims(input_dim)

        enc, d = [], input_dim
        for h in hidden_dims:
            enc += [nn.Linear(d, h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(0.1)]
            d = h
        enc += [nn.Linear(d, latent_dim)]
        self.encoder = nn.Sequential(*enc)

        dec, d = [], latent_dim
        for h in reversed(hidden_dims):
            dec += [nn.Linear(d, h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(0.1)]
            d = h
        dec += [nn.Linear(d, input_dim)]
        self.decoder = nn.Sequential(*dec)

        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.hidden_dims = hidden_dims

    def forward(self, x):
        z = self.encoder(x)
        xhat = self.decoder(z)
        return xhat, z

    @torch.no_grad()
    def score(self, x):
        xhat, _ = self.forward(x)
        return ((x - xhat) ** 2).mean(dim=1)


def train_ae(model, X_tr, X_va, fname):
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    mse = nn.MSELoss()
    amp_scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    tr_ld = DataLoader(
        TensorDataset(torch.tensor(X_tr, dtype=torch.float32)),
        batch_size=BATCH_SIZE, shuffle=True,
        drop_last=False, pin_memory=True, num_workers=2,
    )
    va_ld = DataLoader(
        TensorDataset(torch.tensor(X_va, dtype=torch.float32)),
        batch_size=BATCH_SIZE, shuffle=False,
        pin_memory=True, num_workers=2,
    )

    tr_hist, va_hist = [], []
    best_va, best_state = np.inf, None
    patience_counter = 0

    for ep in tqdm(range(1, EPOCHS + 1), desc=f"  {fname}", leave=False):
        model.train()
        tl = 0.0
        for (xb,) in tr_ld:
            xb = xb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                xhat, _ = model(xb)
                loss = mse(xhat, xb)
            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            amp_scaler.step(opt)
            amp_scaler.update()
            tl += loss.item()
        tl /= len(tr_ld)

        model.eval()
        vl = 0.0
        with torch.no_grad():
            for (xb,) in va_ld:
                xb = xb.to(DEVICE, non_blocking=True)
                with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                    xhat, _ = model(xb)
                    vl += mse(xhat, xb).item()
        vl /= max(len(va_ld), 1)
        sch.step()

        tr_hist.append(tl)
        va_hist.append(vl)

        if vl < best_va - MIN_DELTA:
            best_va = vl
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if EARLY_STOP and patience_counter >= PATIENCE:
            print(f"[{fname}] Early stopping at epoch {ep}")
            break

    if best_state:
        model.load_state_dict(best_state)
    return model, tr_hist, va_hist


@torch.no_grad()
def get_scores(model, X):
    model.eval()
    ds = DataLoader(
        TensorDataset(torch.tensor(X, dtype=torch.float32)),
        batch_size=2048, shuffle=False, pin_memory=True,
    )
    sc = []
    for (xb,) in ds:
        xb = xb.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            sc.append(model.score(xb).cpu())
    return torch.cat(sc).numpy()


def compute_metrics(scores, y_true, threshold):
    pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return dict(
        Precision=precision_score(y_true, pred, zero_division=0),
        Recall=recall_score(y_true, pred, zero_division=0),
        F1=f1_score(y_true, pred, zero_division=0),
        FPR=fp / (fp + tn) if (fp + tn) > 0 else 0.0,
        FNR=fn / (fn + tp) if (fn + tp) > 0 else 0.0,
        ROC_AUC=roc_auc_score(y_true, scores) if len(np.unique(y_true)) > 1 else 0.0,
        PR_AUC=average_precision_score(y_true, scores) if len(np.unique(y_true)) > 1 else 0.0,
        TP=int(tp), FP=int(fp), TN=int(tn), FN=int(fn),
        Threshold=float(threshold),
    )


def find_best_threshold(scores, y_true):
    best_f1, best_thr = 0.0, np.percentile(scores, 80)
    for thr in np.percentile(scores, np.arange(70, 99.9, 0.5)):
        pred = (scores >= thr).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, thr
    return best_thr


# Train and evaluate each embedding dataset
pt_files = sorted(f for f in os.listdir(EMB_DIR) if f.endswith("_emb.pt"))
all_results = []
all_z_scores = {}
all_labels = {}

for pt_name in pt_files:
    stem = pt_name.replace("_emb.pt", "")
    d = torch.load(os.path.join(EMB_DIR, pt_name), map_location="cpu", weights_only=False)
    X = d["X"].numpy() if torch.is_tensor(d["X"]) else np.asarray(d["X"], dtype=np.float32)
    y = d["y"].numpy() if torch.is_tensor(d["y"]) else np.asarray(d["y"])

    n, D = X.shape
    n_tr = int(n * TRAIN_RATIO)

    # Train on normal samples in the first 80%; validate on the last 15% of them
    X_tr_all = X[:n_tr]
    y_tr_all = y[:n_tr]
    X_normal = X_tr_all[y_tr_all == 0]
    n_val = max(int(len(X_normal) * 0.15), 5)
    X_tr = X_normal[:-n_val]
    X_va = X_normal[-n_val:]

    # Test on the last 20% with insider labels 1–4 mapped to 1
    X_test = X[n_tr:]
    y_test_raw = y[n_tr:]
    y_test = binarize_insider_labels(y_test_raw)

    if len(X_tr) < 10 or len(X_test) == 0:
        print(f"[{stem}] Skipped: insufficient training or test samples")
        continue

    model = Autoencoder(D).to(DEVICE)
    model, tr_h, va_h = train_ae(model, X_tr, X_va, stem)
    scores = get_scores(model, X_test)

    # Z-score normalization for the original ensemble
    norm_scores = scores[y_test == 0]
    if len(norm_scores) > 0:
        mu_s = norm_scores.mean()
        sig_s = max(norm_scores.std(), 1e-9)
        z_scores = (scores - mu_s) / sig_s
    else:
        z_scores = scores.copy()

    all_z_scores[stem] = z_scores
    all_labels[stem] = y_test

    thr = find_best_threshold(scores, y_test)
    metrics = compute_metrics(scores, y_test, thr)

    torch.save({
        "model_state": model.state_dict(),
        "input_dim": D, "latent_dim": model.latent_dim,
        "metrics": metrics, "train_loss": tr_h, "val_loss": va_h,
        "data_source": os.path.join(EMB_DIR, pt_name),
        "n_train": len(X_tr), "n_test": len(X_test),
        "label_mapping": "insider 1,2,3,4 → 1",
    }, os.path.join(OUTPUT_DIR, f"{stem}_model.pt"))

    all_results.append({
        "file": stem,
        **{k: round(metrics[k], 4) for k in
           ["Precision", "Recall", "F1", "FPR", "FNR", "ROC_AUC", "PR_AUC"]},
    })
    print(f"[{stem}] F1={metrics['F1']:.4f}, ROC-AUC={metrics['ROC_AUC']:.4f}, "
          f"PR-AUC={metrics['PR_AUC']:.4f}")

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# Max z-score ensemble for datasets with matching row counts
# Note: Equal row counts do not ensure that rows represent the same events.
groups = {}
for stem, z in all_z_scores.items():
    groups.setdefault(len(z), []).append(stem)

for n_rows, stems in groups.items():
    if len(stems) < 2:
        continue

    z_combined = np.stack([all_z_scores[s] for s in stems], axis=0).max(axis=0)
    y_combined = all_labels[stems[0]]
    m_ens = compute_metrics(
        z_combined, y_combined,
        find_best_threshold(z_combined, y_combined),
    )
    all_results.append({
        "file": f"ENSEMBLE({len(stems)})",
        **{k: round(m_ens[k], 4) for k in
           ["Precision", "Recall", "F1", "FPR", "FNR", "ROC_AUC", "PR_AUC"]},
    })

if all_results:
    result_df = pd.DataFrame(all_results).set_index("file")
    print("\nTEST RESULTS")
    print(result_df.to_string())
    result_df.to_csv(os.path.join(OUTPUT_DIR, "summary_results.csv"))

print(f"Results saved to {OUTPUT_DIR}")
